# ТОП-20 к собственному капиталу — сбор ряда и анализ уровня

Обновлённая версия `find_files.ipynb`. Запускать ячейки подряд.

## Что исправлено против прежней версии

1. **`drop_duplicates(subset=['Report Date'])` терял файлы.** pandas считает все
   `NaT` равными между собой, поэтому все файлы, у которых дата в имени не
   распозналась, схлопывались в одну строку и молча исчезали. Теперь такие файлы
   выносятся в `01_no_date.xlsx` для разбора, а не удаляются.

2. **Позиционный разбор `usecols="C,G"` + `tail(6)` ненадёжен.** Если в файле есть
   строка с подписью, но без значения, `dropna(how='all')` её оставляет, хвост
   сдвигается, и все шесть величин приезжают не на свои места — без единого
   признака ошибки. На интервале в несколько лет форма отчёта почти наверняка
   менялась. Теперь показатель ищется по подписи, а в `02_extraction_audit.xlsx`
   пишется, из какой строки и какого столбца взято каждое значение.

3. **`clean_numeric` умножала на 100 всё подряд**, включая ИТОГО, ИТОГО за вычетом
   и СК — суммы в тенге. После деления на `1e9` подпись «в млрд» была неверна
   ровно в 100 раз. Проценты и суммы теперь обрабатываются разными функциями.

Отдельно: прежний прогон показал в столбце лимита значения `[0.95, 2.5]`, то есть
95 % и 250 %. Автоматически это не «чинится» — ячейка 4 печатает все встреченные
значения лимита с датами, чтобы было видно, откуда взялось второе.

## Что добавлено

- Сверка оперативного расчёта с квартальным отчётом об уровнях риск-аппетита
  (форма 50). Если они систематически расходятся — это разные периметры, и
  пересматривать уровень до выяснения нельзя.
- Пересчёт доли из компонентов и список файлов, где расхождение больше 1 пп.
- Частота наблюдений и пропуски в ряде.
- Эпизоды нахождения выше уровня: когда, сколько дней, какой максимум.
- Подбор сигнального уровня с подсчётом ложных срабатываний и запаса времени
  до пробоя.
- Разложение: рост доли шёл за счёт числителя или знаменателя.

## Что прислать после прогона

Достаточно `05_summary.json` и вывода ячейки 5. При расхождениях в сверке —
ещё `03_reconciliation_vs_RA.xlsx`.


In [ ]:
# =============================================================================
# ЯЧЕЙКА 1 — КОНФИГУРАЦИЯ
# =============================================================================
from pathlib import Path
import re

NETWORK_PATH = r"R:\ukr(Y)\20 крупных заемщиков"
OUT_DIR      = Path(r"C:\project_mz\limits\out")

NAME_KEYWORD    = "отправк"          # фильтр по имени файла
EXCLUDE_IN_PATH = ["расчёт", "расчет"]  # исключаемые подстроки в полном пути

# Несколько форматов даты в имени файла. Порядок важен: сначала более длинные.
DATE_PATTERNS = [
    (re.compile(r'(\d{2})[_.\- ](\d{2})[_.\- ](\d{4})'), "dmy"),   # 01.07.2026
    (re.compile(r'(\d{4})[_.\- ](\d{2})[_.\- ](\d{2})'), "ymd"),   # 2026-07-01
    (re.compile(r'(\d{2})[_.\- ](\d{2})[_.\- ](\d{2})(?!\d)'), "dmy2"),  # 01.07.26
]

# Эталонные точки из отчёта об уровнях риск-аппетита (форма 50, лист «Выводы»).
# Нужны для сверки: совпадает ли оперативный недельный расчёт с квартальным РА.
RA_REFERENCE = {
    "2024-07-01": 73.7390, "2024-10-01": 74.1839, "2025-01-01": 69.5187,
    "2025-04-01": 74.0684, "2025-07-01": 76.7365, "2025-10-01": 77.0724,
    "2026-01-01": 75.6193, "2026-04-01": 92.6568, "2026-07-01": 101.5041,
}

RA_LIMIT = 95.0   # действующий уровень риск-аппетита, Приложение № 3 к Политике

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Конфигурация загружена. Выгрузка ->", OUT_DIR)

In [ ]:
# =============================================================================
# ЯЧЕЙКА 2 — ИНВЕНТАРИЗАЦИЯ ФАЙЛОВ
#
# Что исправлено против прежней версии:
#  1. drop_duplicates по 'Report Date' схлопывал ВСЕ файлы с непрочитанной датой
#     в одну строку (pandas считает NaT равными между собой). Теперь файлы без
#     даты не удаляются, а выносятся в отдельный список для разбора.
#  2. Распознаётся три формата даты, а не один.
#  3. Пишется полный журнал: что найдено, что отброшено и по какой причине.
# =============================================================================
import pandas as pd
from datetime import datetime, date

def parse_date_from_name(name: str):
    """Возвращает (date | None, распознанная_подстрока, формат)."""
    for pat, kind in DATE_PATTERNS:
        m = pat.search(name)
        if not m:
            continue
        try:
            if kind == "dmy":
                d, mth, y = int(m.group(1)), int(m.group(2)), int(m.group(3))
            elif kind == "ymd":
                y, mth, d = int(m.group(1)), int(m.group(2)), int(m.group(3))
            else:  # dmy2 — двузначный год
                d, mth, y = int(m.group(1)), int(m.group(2)), 2000 + int(m.group(3))
            return date(y, mth, d), m.group(0), kind
        except ValueError:
            continue          # 32.13.2026 и подобное — пробуем следующий шаблон
    return None, None, None

base_dir = Path(NETWORK_PATH)
rows, skipped = [], []

if not base_dir.exists():
    raise SystemExit(f"Сетевой путь недоступен: {NETWORK_PATH}")

print(f"Сканирую {NETWORK_PATH} ...")
for fp in base_dir.rglob('*'):
    if not fp.is_file():
        continue
    reason = None
    if fp.suffix.lower() != '.xlsx':          reason = "не .xlsx"
    elif fp.name.startswith('~$'):            reason = "временный файл Excel"
    elif any(x in str(fp).lower() for x in EXCLUDE_IN_PATH): reason = "путь содержит «расчёт»"
    elif NAME_KEYWORD.lower() not in fp.name.lower():        reason = "нет ключевого слова"
    if reason:
        if fp.suffix.lower() == '.xlsx' and not fp.name.startswith('~$'):
            skipped.append({"File Name": fp.name, "Reason": reason, "Full Path": str(fp)})
        continue
    try:
        st = fp.stat()
    except (PermissionError, FileNotFoundError, OSError) as e:
        skipped.append({"File Name": fp.name, "Reason": f"нет доступа: {e}", "Full Path": str(fp)})
        continue
    rep_date, raw, kind = parse_date_from_name(fp.name)
    rows.append({
        "Report Date": rep_date,
        "Raw Date String": raw or "",
        "Date Format": kind or "",
        "File Name": fp.name,
        "Modified Date": datetime.fromtimestamp(st.st_mtime),
        "Size KB": round(st.st_size / 1024, 1),
        "Folder": str(fp.parent),
        "Full Path": str(fp),
    })

inv = pd.DataFrame(rows)
print(f"Подходящих файлов: {len(inv)}   отброшено .xlsx: {len(skipped)}")

no_date = inv[inv["Report Date"].isna()].copy()
dated   = inv[inv["Report Date"].notna()].copy()
print(f"  с распознанной датой: {len(dated)}   без даты: {len(no_date)}")

# Дубликаты по дате: оставляем самый свежий по времени изменения, остальные — в журнал
dated = dated.sort_values(["Report Date", "Modified Date"], ascending=[False, False])
dups = dated[dated.duplicated(subset=["Report Date"], keep="first")].copy()
files_df = dated.drop_duplicates(subset=["Report Date"], keep="first").reset_index(drop=True)

print(f"  уникальных отчётных дат: {len(files_df)}   отброшено дублей: {len(dups)}")
if len(files_df):
    print(f"  период: {files_df['Report Date'].min()} .. {files_df['Report Date'].max()}")

files_df.to_excel(OUT_DIR / "01_inventory.xlsx", index=False)
if len(no_date): no_date.to_excel(OUT_DIR / "01_no_date.xlsx", index=False)
if len(dups):    dups.to_excel(OUT_DIR / "01_duplicates.xlsx", index=False)
if skipped:      pd.DataFrame(skipped).to_excel(OUT_DIR / "01_skipped.xlsx", index=False)
print("Записано в", OUT_DIR)

In [ ]:
# =============================================================================
# ЯЧЕЙКА 3 — ИЗВЛЕЧЕНИЕ ПОКАЗАТЕЛЕЙ
#
# Главное изменение. Прежняя версия брала последние 6 непустых строк столбцов
# C и G и раскладывала их по позициям. Это ломается молча: если в файле есть
# строка с подписью, но без значения, dropna(how='all') её сохраняет, хвост
# смещается, и все шесть величин приезжают не на свои места. На интервале в
# несколько лет форма отчёта почти наверняка менялась — проверить это по
# позиционному разбору невозможно.
#
# Теперь: показатель ищется ПО ПОДПИСИ в столбце-метке, значение берётся из
# той же строки. Для каждого файла в журнал пишется, какая строка и какой
# столбец сработали, и полный текст подписи. Позиционный разбор остаётся
# запасным вариантом и помечается в поле Method.
# =============================================================================
import pandas as pd, numpy as np, re, time

LABEL_COL_CANDIDATES = [2, 1, 0, 3]      # C, B, A, D — где искать подписи
VALUE_COL_CANDIDATES = [6, 5, 7, 4, 8]   # G, F, H, E, I — где искать значения

def norm(s):
    return re.sub(r'\s+', ' ', str(s)).strip().lower().replace('ё', 'е')

# Порядок важен: «итого за вычетом» проверяется раньше «итого»
METRICS = [
    ("total_net", "ИТОГО за вычетом денег",       lambda s: "итого" in s and "вычет" in s),
    ("total_raw", "ИТОГО",                        lambda s: s.startswith("итого") and "вычет" not in s and "ск" not in s),
    ("equity",    "Итого СК",                     lambda s: "ск" in s and ("итого" in s or "капитал" in s)),
    ("ratio",     "Доля ТОР-20 к СК",             lambda s: "доля" in s and "ск" in s),
    ("limit",     "Утверждённый лимит по ТОР-20", lambda s: "лимит" in s),
    ("gap",       "Отклонение факта от плана",    lambda s: "отклонен" in s),
]

def to_number(v):
    if v is None or (isinstance(v, float) and np.isnan(v)): return np.nan
    if isinstance(v, str):
        v = v.replace('\xa0', '').replace(' ', '').replace('%', '').replace(',', '.').strip()
        if v in ('', '-', '—'): return np.nan
        try: v = float(v)
        except ValueError: return np.nan
    try: return float(v)
    except (TypeError, ValueError): return np.nan

results, audit, errors = [], [], []
t0 = time.time()

for i, row in files_df.iterrows():
    fpath, rdate, fname = row["Full Path"], row["Report Date"], row["File Name"]
    try:
        xl = pd.ExcelFile(fpath)
        best = None
        for sheet in xl.sheet_names:
            raw = xl.parse(sheet, header=None)
            if raw.empty: continue
            for lc in LABEL_COL_CANDIDATES:
                if lc >= raw.shape[1]: continue
                labels = raw[lc].map(norm)
                hits = {}
                for key, _title, test in METRICS:
                    m = labels[labels.map(lambda s: bool(s) and s != 'nan' and test(s))]
                    if len(m): hits[key] = m.index[-1]   # последнее вхождение — итоговый блок
                if best is None or len(hits) > len(best[2]):
                    best = (sheet, lc, hits, raw)
                if len(hits) == len(METRICS): break
            if best and len(best[2]) == len(METRICS): break

        if best is None:
            raise ValueError("не удалось прочитать ни один лист")
        sheet, label_col, hits, raw = best

        rec = {"Report Date": rdate, "File Name": fname, "Sheet": sheet,
               "Label Col": label_col, "Method": "по подписи",
               "Labels Found": len(hits), "Full Path": fpath}
        aud = dict(rec)

        for key, title, _ in METRICS:
            val, used_col, used_row, lbl = np.nan, None, None, ""
            if key in hits:
                r = hits[key]; used_row = int(r)
                lbl = str(raw.iat[r, label_col])
                for vc in VALUE_COL_CANDIDATES:
                    if vc < raw.shape[1]:
                        cand = to_number(raw.iat[r, vc])
                        if not np.isnan(cand):
                            val, used_col = cand, vc
                            break
            rec[key] = val
            aud[f"{key}__row"] = used_row
            aud[f"{key}__col"] = used_col
            aud[f"{key}__label"] = lbl

        # Запасной вариант: если по подписям нашлось меньше половины показателей
        if len(hits) < 3:
            sub = raw[[label_col, VALUE_COL_CANDIDATES[0]]].dropna(how="all").tail(6)
            vals = [to_number(v) for v in sub[VALUE_COL_CANDIDATES[0]].tolist()]
            while len(vals) < 6: vals.insert(0, np.nan)
            # Прежний порядок строк в отчёте: ИТОГО, ИТОГО за вычетом, СК, доля, лимит, отклонение
            positional_order = ["total_raw", "total_net", "equity", "ratio", "limit", "gap"]
            for key, v in zip(positional_order, vals):
                if np.isnan(rec.get(key, np.nan)): rec[key] = v
            rec["Method"] = "позиционно (запасной)"
            aud["Method"] = rec["Method"]

        results.append(rec); audit.append(aud)
    except Exception as e:
        errors.append({"File Name": fname, "Report Date": rdate, "Error": str(e), "Full Path": fpath})

    if (i + 1) % 50 == 0:
        print(f"  обработано {i+1}/{len(files_df)}  ({time.time()-t0:.0f} c)")

data = pd.DataFrame(results).sort_values("Report Date").reset_index(drop=True)
aud_df = pd.DataFrame(audit)
print(f"\nИзвлечено: {len(data)}   ошибок: {len(errors)}   время: {time.time()-t0:.0f} c")
if len(data):
    print("Метод разбора:"); print(data["Method"].value_counts().to_string())
    print("Найдено подписей на файл:"); print(data["Labels Found"].value_counts().sort_index().to_string())

data.to_excel(OUT_DIR / "02_metrics_raw.xlsx", index=False)
aud_df.to_excel(OUT_DIR / "02_extraction_audit.xlsx", index=False)
if errors: pd.DataFrame(errors).to_excel(OUT_DIR / "02_errors.xlsx", index=False)
print("Записано в", OUT_DIR)

In [ ]:
# =============================================================================
# ЯЧЕЙКА 4 — НОРМАЛИЗАЦИЯ И КОНТРОЛЬ КАЧЕСТВА
#
# Что исправлено: прежняя clean_numeric умножала на 100 ВСЁ, включая ИТОГО,
# ИТОГО за вычетом и СК — то есть абсолютные суммы в тенге. После деления на
# 1e9 подпись «в млрд» была неверна ровно в 100 раз. Проценты и суммы теперь
# обрабатываются разными функциями.
#
# Значение лимита не «чинится» автоматически: печатается распределение всех
# встреченных значений с датами, чтобы аномалии были видны, а не сглажены.
# =============================================================================
import pandas as pd, numpy as np

df = data.copy()
df["Report Date"] = pd.to_datetime(df["Report Date"], errors="coerce")
df = df.dropna(subset=["Report Date"]).sort_values("Report Date").reset_index(drop=True)

def as_percent(v):
    """Доли и лимиты: 0.95 -> 95.0; 95 -> 95.0. Порог 1.5 отделяет доли от процентов."""
    if pd.isna(v): return np.nan
    return v * 100 if abs(v) <= 1.5 else v

df["ratio_pct"] = df["ratio"].apply(as_percent)
df["limit_pct"] = df["limit"].apply(as_percent)
df["gap_pct"]   = df["gap"].apply(as_percent)
for c in ["total_raw", "total_net", "equity"]:
    df[c + "_amt"] = pd.to_numeric(df[c], errors="coerce")   # без домножения
df["cash_collateral_amt"] = df["total_raw_amt"] - df["total_net_amt"]

# --- контроль 1: пересчёт доли из компонентов ---
df["ratio_recalc"] = 100 * df["total_raw_amt"] / df["equity_amt"]
df["ratio_diff"]   = df["ratio_pct"] - df["ratio_recalc"]

# --- контроль 2: какие значения лимита встречаются и когда ---
print("=" * 70)
print("ЗНАЧЕНИЯ ЛИМИТА, ВСТРЕЧЕННЫЕ В ФАЙЛАХ")
print("=" * 70)
lim = df.dropna(subset=["limit_pct"]).groupby(["limit", "limit_pct"]).agg(
    файлов=("File Name", "count"),
    с_даты=("Report Date", "min"),
    по_дату=("Report Date", "max")).reset_index()
lim = lim.rename(columns={"limit": "как в файле", "limit_pct": "после нормализации, %"})
print(lim.to_string(index=False))
print("\nЗначение 0.95 -> 95%. Значение вида 2.5 не преобразуется (порог 1.5) и остаётся как есть —")
print("это сигнал, что в части файлов лимит записан в другой шкале или строка прочитана не та.")
print("\nМоменты смены лимита:")
ch = df.dropna(subset=["limit_pct"]).copy()
ch["prev"] = ch["limit_pct"].shift(1)
ch = ch[ch["limit_pct"] != ch["prev"]]
print(ch[["Report Date", "limit_pct", "prev", "File Name"]].to_string(index=False))

# --- контроль 3: расхождение доли с пересчётом ---
bad = df[df["ratio_diff"].abs() > 1.0].dropna(subset=["ratio_diff"])
print("\n" + "=" * 70)
print(f"ФАЙЛЫ, ГДЕ ДОЛЯ РАСХОДИТСЯ С ПЕРЕСЧЁТОМ БОЛЕЕ ЧЕМ НА 1 пп: {len(bad)}")
print("=" * 70)
if len(bad):
    print(bad[["Report Date", "ratio_pct", "ratio_recalc", "ratio_diff", "File Name"]].head(25).to_string(index=False))

# --- контроль 4: пропуски и частота наблюдений ---
print("\n" + "=" * 70)
print("ПОЛНОТА И ЧАСТОТА")
print("=" * 70)
print(f"Наблюдений: {len(df)}   период: {df['Report Date'].min().date()} .. {df['Report Date'].max().date()}")
gaps = df["Report Date"].diff().dt.days.dropna()
if len(gaps):
    print(f"Шаг между отчётами, дней: медиана {gaps.median():.0f}, среднее {gaps.mean():.1f}, макс {gaps.max():.0f}")
    print("Распределение шага:"); print(gaps.value_counts().sort_index().head(12).to_string())
    big = df.loc[gaps[gaps > 21].index, ["Report Date"]]
    if len(big): print(f"\nПропуски свыше 21 дня — {len(big)} шт., ближайшие даты после разрыва:");  print(big.head(15).to_string(index=False))
print("\nПропущенные значения по столбцам:")
print(df[["ratio_pct","limit_pct","total_raw_amt","total_net_amt","equity_amt"]].isna().sum().to_string())

# --- контроль 5: сверка с квартальным отчётом РА ---
print("\n" + "=" * 70)
print("СВЕРКА С ОТЧЁТОМ ОБ УРОВНЯХ РИСК-АППЕТИТА (форма 50)")
print("=" * 70)
rec_rows = []
for ds, ra_val in RA_REFERENCE.items():
    target = pd.Timestamp(ds)
    sub = df.dropna(subset=["ratio_pct"])
    if not len(sub): continue
    idx = (sub["Report Date"] - target).abs().idxmin()
    near = sub.loc[idx]
    rec_rows.append({
        "Дата РА": target.date(), "Значение РА": ra_val,
        "Ближайший файл": near["Report Date"].date(),
        "Дней разницы": int(abs((near["Report Date"] - target).days)),
        "Значение из файла": round(near["ratio_pct"], 4),
        "Расхождение, пп": round(near["ratio_pct"] - ra_val, 4),
    })
rec_df = pd.DataFrame(rec_rows)
print(rec_df.to_string(index=False))
print("\nЕсли расхождения систематические — оперативный расчёт и метрика риск-аппетита")
print("считаются на разных периметрах, и это надо зафиксировать до пересмотра уровня.")

df.to_excel(OUT_DIR / "03_series_clean.xlsx", index=False)
df.to_csv(OUT_DIR / "03_series_clean.csv", index=False, encoding="utf-8-sig")
rec_df.to_excel(OUT_DIR / "03_reconciliation_vs_RA.xlsx", index=False)
print("\nЗаписано в", OUT_DIR)

In [ ]:
# =============================================================================
# ЯЧЕЙКА 5 — АНАЛИЗ ДЛЯ ПЕРЕСМОТРА УРОВНЯ РИСК-АППЕТИТА
#
# Ради этой ячейки всё остальное и делается. По квартальному отчёту у нас было
# девять точек; здесь ряд на порядок длиннее, и на нём можно посчитать то, что
# на девяти точках посчитать нельзя: устойчивую волатильность, реальные
# пересечения уровня, длительность нахождения выше него и качество любого
# кандидата в сигнальный уровень.
# =============================================================================
import pandas as pd, numpy as np

s = df.dropna(subset=["ratio_pct"]).set_index("Report Date")["ratio_pct"].sort_index()
print(f"Ряд: {len(s)} наблюдений, {s.index.min().date()} .. {s.index.max().date()}")
print(f"Уровень риск-аппетита: {RA_LIMIT}%\n")

# ---- 1. Описательная статистика по годам ----
print("=" * 78); print("ПО ГОДАМ"); print("=" * 78)
yr = s.groupby(s.index.year).agg(["count", "min", "mean", "max", "std"]).round(2)
yr["дней выше лимита"] = s.groupby(s.index.year).apply(lambda x: int((x > RA_LIMIT).sum()))
print(yr.to_string())

# ---- 2. Волатильность приращений ----
d = s.diff().dropna()
print("\n" + "=" * 78); print("ПРИРАЩЕНИЯ МЕЖДУ СОСЕДНИМИ ОТЧЁТАМИ, пп"); print("=" * 78)
print(f"σ по всему ряду: {d.std():.3f}   |Δ| медиана: {d.abs().median():.3f}   max: {d.abs().max():.3f}")
print("σ по годам:"); print(d.groupby(d.index.year).std().round(3).to_string())
print("\n10 крупнейших скачков:")
print(d.reindex(d.abs().sort_values(ascending=False).index).head(10).round(3).to_string())

# ---- 3. Пересечения уровня ----
above = s > RA_LIMIT
grp = (above != above.shift()).cumsum()
episodes = []
for _, blk in s.groupby(grp):
    if blk.iloc[0] > RA_LIMIT:
        episodes.append({"начало": blk.index.min().date(), "конец": blk.index.max().date(),
                         "наблюдений": len(blk), "дней": (blk.index.max() - blk.index.min()).days + 1,
                         "максимум": round(blk.max(), 2)})
print("\n" + "=" * 78); print(f"ЭПИЗОДЫ ВЫШЕ УРОВНЯ {RA_LIMIT}%: {len(episodes)}"); print("=" * 78)
if episodes:
    ep = pd.DataFrame(episodes); print(ep.to_string(index=False))
    print(f"\nВсего наблюдений выше уровня: {int(above.sum())} из {len(s)} ({100*above.mean():.1f}%)")
else:
    print("Уровень не пробивался ни разу за весь период.")

# ---- 4. Подбор сигнального уровня ----
# Критерий: сигнал должен срабатывать ДО пробоя и не шуметь в спокойные периоды.
print("\n" + "=" * 78); print("КАНДИДАТЫ В СИГНАЛЬНЫЙ УРОВЕНЬ"); print("=" * 78)
first_breach = s[s > RA_LIMIT].index.min() if above.any() else None
rows = []
for thr in [75, 78, 80, 82, 85, 87, 88, 90, 92]:
    hit = s > thr
    if not hit.any():
        rows.append({"уровень": thr, "срабатываний": 0, "первое": "—", "предупреждение, дней": "—",
                     "ложных до пробоя": 0, "доля времени в жёлтой": 0.0})
        continue
    first_hit = hit[hit].index.min()
    warn = (first_breach - first_hit).days if first_breach is not None and first_hit < first_breach else 0
    # «Ложное» — срабатывание, после которого в течение 180 дней пробоя не было
    false_n = 0
    for t in s.index[hit]:
        window = s[(s.index > t) & (s.index <= t + pd.Timedelta(days=180))]
        if len(window) and window.max() <= RA_LIMIT: false_n += 1
    rows.append({"уровень": thr, "срабатываний": int(hit.sum()),
                 "первое": first_hit.date(),
                 "предупреждение, дней": warn,
                 "ложных до пробоя": false_n,
                 "доля времени в жёлтой": round(100 * ((s > thr) & (s <= RA_LIMIT)).mean(), 1)})
cand = pd.DataFrame(rows)
print(cand.to_string(index=False))
print("\n«Ложное» = сигнал, после которого пробоя не случилось в течение 180 дней.")
print("«Доля времени в жёлтой» выше 30–40% означает, что зона перестаёт быть сигналом:")
print("постоянное нахождение в ней говорит о неверной калибровке самого уровня, а не о риске.")

# ---- 5. Разложение: числитель или знаменатель ----
print("\n" + "=" * 78); print("ЧТО ДВИГАЛО ПОКАЗАТЕЛЬ"); print("=" * 78)
comp = df.dropna(subset=["ratio_pct", "total_raw_amt", "equity_amt"]).copy()
if len(comp) > 1:
    comp = comp.set_index("Report Date").sort_index()
    ann = comp.resample("QE").last()[["ratio_pct", "total_raw_amt", "equity_amt", "cash_collateral_amt"]]
    ann["Δ доля, пп"]   = ann["ratio_pct"].diff().round(2)
    ann["Δ ТОП-20, %"]  = (100 * ann["total_raw_amt"].pct_change()).round(2)
    ann["Δ СК, %"]      = (100 * ann["equity_amt"].pct_change()).round(2)
    print(ann.tail(16).round(2).to_string())
    print("\nЕсли доля растёт при падающем СК — это вопрос к капиталу, а не к концентрации,")
    print("и меры реагирования должны быть другими.")

# ---- 6. Выгрузка компактного ряда ----
out = df[["Report Date", "ratio_pct", "limit_pct", "total_raw_amt", "total_net_amt",
          "equity_amt", "cash_collateral_amt", "Method", "File Name"]].copy()
out.to_csv(OUT_DIR / "04_top20_series_for_analysis.csv", index=False, encoding="utf-8-sig")
cand.to_csv(OUT_DIR / "04_signal_level_candidates.csv", index=False, encoding="utf-8-sig")
if episodes: pd.DataFrame(episodes).to_csv(OUT_DIR / "04_breach_episodes.csv", index=False, encoding="utf-8-sig")

# Компактная сводка для передачи — помещается в сообщение
summary = {
    "наблюдений": len(s), "с": str(s.index.min().date()), "по": str(s.index.max().date()),
    "медианный шаг, дней": float(pd.Series(s.index).diff().dt.days.median()),
    "мин": round(float(s.min()), 2), "медиана": round(float(s.median()), 2), "макс": round(float(s.max()), 2),
    "σ приращений, пп": round(float(d.std()), 3),
    "макс скачок, пп": round(float(d.abs().max()), 3),
    "эпизодов выше лимита": len(episodes),
    "наблюдений выше лимита": int(above.sum()),
    "значения лимита в файлах": sorted(df["limit_pct"].dropna().unique().tolist()),
}
import json
(OUT_DIR / "05_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print("\n" + "=" * 78); print("СВОДКА (её достаточно прислать первой)"); print("=" * 78)
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
# =============================================================================
# ЯЧЕЙКА 6 — ГРАФИКИ
#
# Исправлено: абсолютные суммы больше не умножаются на 100, поэтому подпись
# «млрд» теперь соответствует величине. Масштаб определяется по данным, а не
# по порогу в 1e6. На первом графике добавлены уровень риск-аппетита и
# кандидат в сигнальный уровень — чтобы график отвечал на вопрос пересмотра,
# а не только показывал линию.
# =============================================================================
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, matplotlib.dates as mdates, matplotlib.ticker as mtick
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
import numpy as np, pandas as pd

SIGNAL_LEVEL = 85.0   # кандидат; уточняется по таблице из ячейки 5

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"font.size": 11, "axes.labelsize": 12, "axes.titlesize": 13})

v = df.dropna(subset=["Report Date", "ratio_pct"]).sort_values("Report Date")

# ---------- График 1: доля, уровень РА, сигнальный уровень ----------
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.axhspan(RA_LIMIT, max(105, v["ratio_pct"].max() * 1.02), color="#d62728", alpha=0.07, zorder=0)
ax.axhspan(SIGNAL_LEVEL, RA_LIMIT, color="#ffc107", alpha=0.10, zorder=0)

lim_line = v["limit_pct"].ffill()
ax.step(v["Report Date"], lim_line, where="post", color="#d62728", lw=2,
        label=f"Утверждённый уровень (факт из файлов)", zorder=3)
ax.axhline(SIGNAL_LEVEL, color="#b8860b", lw=1.6, ls="--",
           label=f"Кандидат в сигнальный уровень — {SIGNAL_LEVEL:.0f}%", zorder=3)

x = mdates.date2num(v["Report Date"]); y = v["ratio_pct"].values
pts = np.array([x, y]).T.reshape(-1, 1, 2)
seg = np.concatenate([pts[:-1], pts[1:]], axis=1)
cmap = mcolors.LinearSegmentedColormap.from_list("risk", ["#2ca02c", "#ffc107", "#d62728"])
lc = LineCollection(seg, cmap=cmap, norm=plt.Normalize(vmin=70, vmax=RA_LIMIT + 5), lw=2.2, zorder=4)
lc.set_array(y); ax.add_collection(lc)
ax.plot(v["Report Date"], v["ratio_pct"], alpha=0)
ax.plot([], [], color="#ffc107", lw=2.2, label="Фактическая доля ТОП-20 к СК")

br = v[v["ratio_pct"] > RA_LIMIT]
if len(br):
    ax.scatter(br["Report Date"], br["ratio_pct"], s=14, color="#d62728", zorder=5,
               label=f"Выше уровня — {len(br)} набл.")

ch = v.dropna(subset=["limit_pct"]).copy(); ch["prev"] = ch["limit_pct"].shift(1)
for _, r in ch[(ch["limit_pct"] != ch["prev"]) & ch["prev"].notna()].iterrows():
    ax.annotate(f"{r['Report Date'].strftime('%d.%m.%Y')}\n{r['limit_pct']:.0f}%",
                xy=(r["Report Date"], r["limit_pct"]), xytext=(8, 16), textcoords="offset points",
                fontsize=8.5, fontweight="bold", color="#800000",
                bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="#d62728", alpha=0.9),
                arrowprops=dict(arrowstyle="->", color="#d62728", shrinkB=4))

ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_title("ТОП-20 к собственному капиталу: факт, уровень риск-аппетита и зоны", fontweight="bold", pad=14)
ax.set_ylabel("Доля к капиталу")
ax.xaxis.set_major_locator(mdates.YearLocator()); ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.legend(loc="upper left", frameon=True, fontsize=9)
plt.tight_layout(); plt.savefig(OUT_DIR / "fig1_ratio_zones.png", dpi=200); plt.close()

# ---------- График 2: абсолютные величины ----------
a = df.dropna(subset=["Report Date"]).sort_values("Report Date")
mx = a["total_raw_amt"].abs().max()
scale, unit = (1e9, "млрд") if mx > 1e9 else ((1e6, "млн") if mx > 1e6 else (1, ""))
fig, (t, b) = plt.subplots(2, 1, figsize=(13, 8), gridspec_kw={"height_ratios": [3, 1.2]}, sharex=True)
t.plot(a["Report Date"], a["total_raw_amt"] / scale, color="#d62728", lw=1.8, label="Задолженность ТОП-20, валовая")
t.plot(a["Report Date"], a["total_net_amt"] / scale, color="#2ca02c", lw=1.8, label="За вычетом денежного обеспечения")
t.plot(a["Report Date"], a["equity_amt"] / scale, color="#7f7f7f", lw=1.4, label="Собственный капитал")
t.set_title("Портфель ТОП-20 и капитал", fontweight="bold", pad=14)
t.set_ylabel(f"Сумма, {unit}"); t.legend(loc="upper left", fontsize=9)
b.plot(a["Report Date"], a["cash_collateral_amt"] / scale, color="#1f77b4", lw=1.4, label="Денежное обеспечение")
b.fill_between(a["Report Date"], a["cash_collateral_amt"] / scale, color="#1f77b4", alpha=0.15)
b.set_ylabel(f"Обеспечение,\n{unit}"); b.legend(loc="upper left", fontsize=9)
b.xaxis.set_major_locator(mdates.YearLocator()); b.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
plt.tight_layout(); plt.subplots_adjust(hspace=0.08)
plt.savefig(OUT_DIR / "fig2_absolute.png", dpi=200); plt.close()

# ---------- График 3: сверка с квартальным отчётом РА ----------
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(v["Report Date"], v["ratio_pct"], color="#17becf", lw=1, alpha=0.6, label="Оперативный расчёт (файлы)")
rr = pd.DataFrame([{"d": pd.Timestamp(k), "v": val} for k, val in RA_REFERENCE.items()]).sort_values("d")
ax.scatter(rr["d"], rr["v"], s=55, color="#d62728", zorder=5, marker="D",
           label="Отчёт об уровнях риск-аппетита (форма 50)")
ax.axhline(RA_LIMIT, color="#d62728", lw=1.4, ls="-", alpha=0.7, label=f"Уровень {RA_LIMIT:.0f}%")
ax.axhline(SIGNAL_LEVEL, color="#b8860b", lw=1.4, ls="--", label=f"Сигнальный {SIGNAL_LEVEL:.0f}%")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(decimals=0))
ax.set_title("Сверка оперативного расчёта с квартальным отчётом риск-аппетита", fontweight="bold", pad=14)
ax.set_ylabel("Доля к капиталу"); ax.legend(loc="upper left", fontsize=9)
ax.set_xlim(pd.Timestamp("2024-01-01"), max(v["Report Date"].max(), rr["d"].max()) + pd.Timedelta(days=30))
plt.tight_layout(); plt.savefig(OUT_DIR / "fig3_reconciliation.png", dpi=200); plt.close()

print("Графики сохранены:")
for n in ["fig1_ratio_zones.png", "fig2_absolute.png", "fig3_reconciliation.png"]:
    print("  ", OUT_DIR / n)